# dko-3dgs — pipeline local

Corre el pipeline completo en esta máquina (`~/dko-3dgs`), reutilizando los scripts del repo:

```
video local → candidates/ (ffmpeg) → select_sharp.py → data/input/
            → run_colmap.sh (CPU) o run_hloc.py (GPU) → data/images + data/sparse/0
            → gaussian-splatting/train.py → output/dko3d/
```

**Requisitos ya instalados en esta máquina:** COLMAP (build CPU), hloc + pycolmap, ffmpeg, y el clon de `gaussian-splatting/` con sus submódulos CUDA.

⚠️ Los directorios `candidates/`, `data/`, `hloc_out/` y `output/` ya tienen los resultados de la corrida anterior. Las celdas marcadas con ⚠️ los sobrescriben — sáltalas si solo quieres re-entrenar.

## 0. Configuración

In [ ]:
from pathlib import Path

ROOT = Path.home() / "dko-3dgs"

VIDEO = "/home/leo/dko-3dgs/dko.mov"

# None = usar TODOS los frames en el SfM; o un número (p. ej. 240) para seleccionar los más nítidos
TARGET_IMAGES = None

# Tope de imágenes para ENTRENAR (train.py carga todo en RAM; con 4K y 30 GB el límite práctico es ~500)
MAX_TRAIN_IMAGES = 500

ITERATIONS = 30000    # entrenamiento completo del paper

# Python del venv con las dependencias (cv2, torch, hloc, pycolmap)
PY = ROOT / "gaussian-splatting" / "env" / "bin" / "python"

print("ROOT:", ROOT)
print("candidates:", len(list((ROOT / 'candidates').glob('*.jpg'))), "frames")
print("data/input:", len(list((ROOT / 'data' / 'input').glob('*.jpg'))), "imágenes")

## 1. ⚠️ Extraer frames del video

Solo si partes de un video nuevo. Sobrescribe `candidates/`.

In [2]:
assert VIDEO, "Define VIDEO en la celda de configuración antes de correr esta celda"

import shutil
cand = ROOT / "candidates"
if cand.exists():
    shutil.rmtree(cand)
cand.mkdir()

!ffmpeg -hide_banner -loglevel error -i "{VIDEO}" -qscale:v 2 "{cand}/%05d.jpg"
print(len(list(cand.glob('*.jpg'))), "frames extraídos")

23916 frames extraídos


## 2. ⚠️ Seleccionar frames nítidos

Usa `select_sharp.py` del repo (varianza del Laplaciano por ventana). Sobrescribe `data/input/`.

In [ ]:
import os, shutil

frames = sorted((ROOT / "candidates").glob("*.jpg"))
n_frames = len(frames)
WIN = 1 if TARGET_IMAGES is None else max(1, n_frames // TARGET_IMAGES)
print(f"{n_frames} frames, ventana = {WIN}")

inp = ROOT / "data" / "input"
if inp.exists():
    shutil.rmtree(inp)

if WIN == 1:
    # todos los frames: hardlinks — instantáneo y no duplica los ~11 GB
    inp.mkdir(parents=True)
    for i, f in enumerate(frames):
        os.link(f, inp / f"{i:05d}.jpg")
    print(f"{n_frames} frames enlazados a data/input")
else:
    !"{PY}" "{ROOT}/select_sharp.py" "{ROOT}/candidates" "{inp}" {WIN}

### Opción A: hloc (GPU) — recomendada

ALIKED + LightGlue + mapper de pycolmap (`run_hloc.py`). Luego se undistorsiona al layout `data/images` + `data/sparse/0` que espera 3DGS.

⏱️ Con TODOS los frames (23916): extracción ~30–60 min y matching ~1.5–2 h en GPU; el **mapper corre en CPU** y puede tomar días — es normal, no lo interrumpas. El log de COLMAP queda en `hloc_out/sfm/colmap.LOG.*`.

In [ ]:
import shutil

hloc_out = ROOT / "hloc_out"
if hloc_out.exists():
    shutil.rmtree(hloc_out)  # ⚠️ borra la corrida anterior de hloc

!cd "{ROOT}" && "{PY}" run_hloc.py

In [ ]:
import pycolmap, shutil

# 1) Submuestrear el modelo SfM a MAX_TRAIN_IMAGES cámaras (uniforme sobre la secuencia).
#    train.py carga TODAS las imágenes del modelo en RAM: 23916 a 4K serían ~2.4 TB.
sfm = ROOT / "hloc_out" / "sfm"
rec = pycolmap.Reconstruction(str(sfm))
reg = sorted(rec.reg_image_ids())
print(f"modelo SfM: {len(reg)} imágenes registradas, {rec.num_points3D()} puntos 3D")

step = max(1, len(reg) // MAX_TRAIN_IMAGES)
keep = set(reg[::step])
for iid in reg:
    if iid not in keep:
        rec.deregister_frame(rec.image(iid).frame_id)

sfm_train = ROOT / "hloc_out" / "sfm_train"
if sfm_train.exists():
    shutil.rmtree(sfm_train)
sfm_train.mkdir()
rec.write(str(sfm_train))
print(f"set de entrenamiento: {len(keep)} cámaras (1 de cada {step})")

# 2) Undistorsionar solo esas al layout que espera 3DGS
data = ROOT / "data"
for d in (data / "images", data / "sparse", data / "stereo", data / "distorted"):
    if d.exists():
        shutil.rmtree(d)

pycolmap.undistort_images(
    output_path=str(data),
    input_path=str(sfm_train),
    image_path=str(data / "input"),
    output_type="COLMAP",
)

sparse0 = data / "sparse" / "0"
sparse0.mkdir(parents=True, exist_ok=True)
for f in (data / "sparse").iterdir():
    if f.is_file():
        shutil.move(str(f), sparse0 / f.name)

# 3) Nube de puntos inicial: usar la del modelo COMPLETO.
#    Al filtrar cámaras casi todos los tracks pierden sus observaciones y los puntos
#    desaparecerían; 3DGS solo lee xyz/rgb de points3D.bin, así que podemos darle
#    todos los puntos del SfM completo (la undistorsión no cambia coordenadas 3D).
shutil.copy(sfm / "points3D.bin", sparse0 / "points3D.bin")
print(sorted(p.name for p in sparse0.iterdir()))

### Opción B: COLMAP (CPU) — horas en esta máquina

SIFT + matching secuencial con loop detection (`run_colmap.sh`). Produce directamente el layout 3DGS en `data/`. Requiere `vocab_tree.bin` en la raíz. El log queda en pantalla; si lo corres fuera del notebook, mejor `./run_colmap.sh > colmap.log 2>&1 &`.

⚠️ Las etapas de extracción y matching son lentas en CPU — no interrumpas aunque parezca colgado.

In [ ]:
# import shutil
# for d in (ROOT/'data'/'images', ROOT/'data'/'sparse', ROOT/'data'/'stereo', ROOT/'data'/'distorted'):
#     if d.exists():
#         shutil.rmtree(d)  # ⚠️ borra la reconstrucción anterior
#
# !cd "{ROOT}" && bash run_colmap.sh

## 4. Entrenar 3DGS

⚠️ Sobrescribe `output/dko3d/`. Cambia `-m` si quieres conservar el modelo anterior (p. ej. `output/dko3d_v2`).

In [ ]:
MODEL_DIR = ROOT / "output" / "dko3d"

# -r 2: media resolución (1080x1920); con --data_device cpu, 500 imágenes ≈ 12 GB RAM
!cd "{ROOT}/gaussian-splatting" && "{PY}" train.py \
    -s "{ROOT}/data" -m "{MODEL_DIR}" -r 2 --data_device cpu \
    --iterations {ITERATIONS} --save_iterations {ITERATIONS} --test_iterations -1

## 5. Resultado

El modelo queda en `output/dko3d/point_cloud/iteration_*/point_cloud.ply`. Ábrelo en [SuperSplat](https://playcanvas.com/supersplat/editor) (arrastra el `.ply` al navegador) o con el visor SIBR del repo de Inria.

In [ ]:
for ply in sorted((ROOT / "output").rglob("point_cloud.ply")):
    print(ply, f"({ply.stat().st_size / 1e6:.1f} MB)")